# Create figurl!

In [1]:
import os
import datajoint as dj
import numpy as np

# change to the upper level folder to detect dj_local_conf.json
if os.path.basename(os.getcwd()) == "decoding":
    os.chdir("..")
dj.config["enable_python_native_blobs"] = True
dj.config.load("dj_local_conf.json")  # load config for database connection info

[2025-12-16 17:35:57,219][INFO]: DataJoint is configured from /home/yshwang/code/Hex-maze-spyglass/dj_local_conf.json


In [2]:
import spyglass
print(spyglass.__file__) # confirm your environment is set up correctly
print(spyglass.__version__) # confirm your spyglass version

/home/yshwang/code/spyglass/src/spyglass/__init__.py
0.5.5a2.dev23+ge4016cd7d.d20251212


In [3]:
from spyglass.decoding.decoding_merge import DecodingOutput

DecodingOutput.SortedSpikesDecodingV1 & {"nwb_file_name": "IM-1478_20220724_.nwb"}  # check if data exists

[2025-12-16 17:36:06,242][INFO]: DataJoint 0.14.6 connected to yshwang@lmf-db.cin.ucsf.edu:3306
INFO:datajoint:DataJoint 0.14.6 connected to yshwang@lmf-db.cin.ucsf.edu:3306


merge_id,nwb_file_name name of the NWB file,unit_filter_params_name,sorted_spikes_group_name,position_group_name,decoding_param_name a name for this set of parameters,encoding_interval descriptive name of this interval list,decoding_interval descriptive name of this interval list,estimate_decoding_params whether to estimate the decoding parameters
4aeda101-ab19-727f-5db1-417f8a791cf4,IM-1478_20220724_.nwb,default_exclusion,sorted_spikes_group,sorted_spikes_pos_group,contfrag_sorted_50chunks,00_r1,epoch0_block1,1


In [4]:
selection_key = {
    "nwb_file_name" : "IM-1478_20220724_.nwb"
}

In [5]:
from spyglass.decoding.v1.sorted_spikes import SortedSpikesDecodingV1

decoding_results = (SortedSpikesDecodingV1 & selection_key).fetch_results()
decoding_results

[2025-12-16 17:36:20,195][WARNING]: Skipped checksum for file with hash: 35a9946c-097a-5a91-dc84-cc71fa1ee442, and path: /stelmo/nwb/analysis/IM-1478_20220724/IM-1478_20220724_8767a4cb-6a44-4379-971d-db6bff5bb214.nc
/home/yshwang/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: UserWarning: Duplicate dimension names present: dimensions {'states'} appear more than once in dims=('states', 'states'). We do not yet support duplicate dimension names, but we do allow initial construction of the object. We recommend you rename the dims immediately to become distinct, as most xarray functionality is likely to fail silently if you do not. To rename the dimensions you will need to set the ``.dims`` attribute of each variable, ``e.g. var.dims=('x0', 'x1')``.
  warnings.warn(
/home/yshwang/miniforge3/envs/spyglass/lib/python3.10/site-packages/xarray/namedarray/core.py:496: UserWarning: Duplicate dimension names present: dimensions {'states'} appear more than onc

<xarray.Dataset> Size: 155GB
Dimensions:                      (time: 3233091, state_bins: 11952, states: 2,
                                  state_ind: 11952, dim_0: 11952)
Coordinates:
  * time                         (time) float64 26MB 12.88 12.89 ... 6.479e+03
  * state_ind                    (state_ind) int32 48kB 0 0 0 0 0 ... 1 1 1 1 1
  * states                       (states) object 16B 'Continuous' 'Fragmented'
    environments                 (states) object 16B ...
    encoding_groups              (states) int32 8B ...
  * state_bins                   (state_bins) object 96kB MultiIndex
  * state                        (state_bins) object 96kB 'Continuous' ... 'F...
  * x_position                   (state_bins) float64 96kB 20.86 20.86 ... 184.6
  * y_position                   (state_bins) float64 96kB 9.599 11.6 ... 151.5
Dimensions without coordinates: dim_0
Data variables:
    acausal_posterior            (time, state_bins) float32 155GB ...
    acausal_state_probabilities  (time, states) float32 26MB ...
    initial_conditions           (dim_0) float64 96kB ...
    discrete_state_transitions   (states, states) float64 32B ...
Attributes:
    marginal_log_likelihoods:  [-4045141.  -4034676.  -4030740.5 -4028935.5 -...

In [9]:
from non_local_detector.model_checking import (
    get_highest_posterior_threshold,
    get_HPD_spatial_coverage,
)
from non_local_detector.visualization import (
    create_interactive_2D_decoding_figurl,
)

(
    position_info,
    position_variable_names,
) = SortedSpikesDecodingV1.fetch_position_info(selection_key)
#results_time = decoding_results.acausal_posterior.isel(intervals=0).time.values
#results_time = decoding_results.time.values #isel(intervals=0).time.values # from stephs
results_time = decoding_results.acausal_posterior.time.values
position_info = position_info.loc[results_time[0] : results_time[-1]]

env = SortedSpikesDecodingV1.fetch_environments(selection_key)[0]
# spike_times, _ = SortedSpikesDecodingV1.fetch_spike_data(selection_key)
spike_times = SortedSpikesDecodingV1.fetch_spike_data(selection_key) # from stephs

create_interactive_2D_decoding_figurl(
    position_time=position_info.index.to_numpy(),
    position=position_info[position_variable_names],
    env=env,
    results=decoding_results,
    # posterior=decoding_results.acausal_posterior.isel(intervals=0)
    # .unstack("state_bins")
    # .sum("state")
    posterior=decoding_results.acausal_posterior
    .unstack("state_bins")
    .sum("state"),
    spike_times=spike_times,
    head_dir=position_info["orientation"],
    speed=position_info["speed"],
)

# url = create_interactive_2D_decoding_figurl(
#     position_time=position_info.index.to_numpy(),
#     position=position_info[position_variable_names],
#     env=env,
#     results=decoding_results,
#     posterior=decoding_results.acausal_posterior.unstack("state_bins").sum("state"),
#     spike_times=spike_times,
#     head_dir=position_info["orientation"],
#     speed=position_info["speed"],
# )
# url

[11:57:11][WARNING] Spyglass: Upsampled position data, frame indices are invalid. Setting add_frame_ind=False
[11:57:14][WARNING] Spyglass: Upsampled position data, frame indices are invalid. Setting add_frame_ind=False


Computing sha1 of /stelmo/nwb/kachery_storage/tmp_B6XRzdFO/file.dat


'https://figurl.org/f?v=npm://@fi-sci/figurl-sortingview@12/dist&d=sha1://256b95984c07c6f1b4e3aa276d38a7af920b9912&label=2D%20Decoding&zone=franklab.default'

In [6]:
from non_local_detector.model_checking import (
    get_highest_posterior_threshold,
    get_HPD_spatial_coverage,
)
from non_local_detector.visualization import (
    create_interactive_2D_decoding_figurl,
)

(
    position_info,
    position_variable_names,
) = SortedSpikesDecodingV1.fetch_position_info(selection_key)
#results_time = decoding_results.acausal_posterior.isel(intervals=0).time.values
results_time = decoding_results.time.values #isel(intervals=0).time.values # from stephs
position_info = position_info.loc[results_time[0] : results_time[-1]]

env = SortedSpikesDecodingV1.fetch_environments(selection_key)[0]
# spike_times, _ = SortedSpikesDecodingV1.fetch_spike_data(selection_key)
spike_times = SortedSpikesDecodingV1.fetch_spike_data(selection_key) # from stephs

create_interactive_2D_decoding_figurl(
    position_time=position_info.index.to_numpy(),
    position=position_info[position_variable_names],
    env=env,
    results=decoding_results,
    # posterior=decoding_results.acausal_posterior.isel(intervals=0)
    # .unstack("state_bins")
    # .sum("state")
    posterior=decoding_results.acausal_posterior
    .unstack("state_bins")
    .sum("state"),
    spike_times=spike_times,
    head_dir=position_info["orientation"],
    speed=position_info["speed"],
)

# url = create_interactive_2D_decoding_figurl(
#     position_time=position_info.index.to_numpy(),
#     position=position_info[position_variable_names],
#     env=env,
#     results=decoding_results,
#     posterior=decoding_results.acausal_posterior.unstack("state_bins").sum("state"),
#     spike_times=spike_times,
#     head_dir=position_info["orientation"],
#     speed=position_info["speed"],
# )
# url

[17:36:28][WARNING] Spyglass: Upsampled position data, frame indices are invalid. Setting add_frame_ind=False
/home/yshwang/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'ndx-franklab-novela' version 0.2.3 because version 0.2.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
[17:36:48][WARNING] Spyglass: Upsampled position data, frame indices are invalid. Setting add_frame_ind=False
/home/yshwang/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'ndx-franklab-novela' version 0.2.3 because version 0.2.0 is already loaded.
  warn("Ignoring cached namespace '%s' version %s because version %s is already loaded."
/home/yshwang/miniforge3/envs/spyglass/lib/python3.10/site-packages/hdmf/spec/namespace.py:535: UserWarning: Ignoring cached namespace 'ndx-franklab-novela' version 0.2.3 because

Computing sha1 of /stelmo/nwb/kachery_storage/tmp_eLv8YBSh/file.dat


'https://figurl.org/f?v=npm://@fi-sci/figurl-sortingview@12/dist&d=sha1://b498f5ba03db05f2e6d181a71485ba36bd2591f1&label=2D%20Decoding&zone=franklab.default'